# Curl in the Citation Graph: Hodge Flow Decomposition for Detecting Citation Manipulation

**Dataset:** OpenAlex Journal Citation Flow Network + Clarivate JCR Suppression Labels (2015–2022)

**Method:** HodgeRank / Hodge decomposition on the directed journal citation graph.
We decompose pairwise citation flows into a **gradient component** (consistent global ranking) and a **curl component** (cyclic inconsistencies). Journals with anomalously high curl residuals are flagged as likely citation manipulators.

**Ground truth:** 40 journals suppressed by Clarivate JCR for citation stacking / excessive self-citation (2018–2022). In this demo we use a 100-example curated subset.

**Key claim:** The curl residual score from Hodge decomposition outperforms simple degree-based features for detecting suppressed journals.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# All packages used here are pre-installed on Colab; install locally to match Colab versions
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'scikit-learn==1.6.1',
         'matplotlib==3.10.0', 'networkx==3.6.1')

In [ ]:
import json
import os
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import networkx as nx
import matplotlib
import matplotlib.pyplot as plt
from collections import defaultdict
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-d509fc-curl-in-the-citation-graph-hodge-flow-de/main/round-1/dataset-1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
examples = data["datasets"][0]["examples"]
print(f"Loaded {len(examples)} citation pair examples")
print(f"Dataset metadata: {data['metadata']['task'][:120]}...")
print(f"\nSuppression rate: {data['metadata']['suppression_rate']}")
n_pos = sum(1 for e in examples if e['metadata_label_i'] == 1)
print(f"Examples with suppressed source journal: {n_pos}/{len(examples)}")

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
# Minimum flow threshold: edges with citation count below this are ignored
MIN_FLOW = 1          # original: 1 (keep all edges)

# Regularization for Hodge least-squares solve (avoids singular Laplacian)
HODGE_REG = 1e-6      # original: 1e-6

# Cross-validation folds for classifier evaluation
N_CV_FOLDS = 3        # original: 5

# Random seed
SEED = 42
np.random.seed(SEED)
# ────────────────────────────────────────────────────────────────────────────

## Step 1: Build the Directed Citation Graph

Each example in the dataset represents a directed citation pair $(i \to j)$ with a weight $C_{ij}$ (number of times journal $i$ cites journal $j$ across all works in 2015–2022).

We convert this into a weighted directed graph $G = (V, E)$ where:
- $V$ = set of all journals that appear in any pair  
- $E$ = directed edges $(i \to j)$ with $C_{ij} > 0$
- Edge weight $f(i \to j) = \log(C_{ij} + 1) - \log(C_{ji} + 1)$: the **log net citation flow** (antisymmetric)

The antisymmetric log-flow $f$ is the input to Hodge decomposition: it captures which direction the net citation pressure runs between every pair of journals.

In [ ]:
# --- Collect all journals and their labels ---
journal_names = {}    # name -> integer index
journal_labels = {}   # name -> ground-truth label (0/1)
edge_counts = {}      # (idx_i, idx_j) -> citation count C[i,j]

def get_idx(name):
    if name not in journal_names:
        journal_names[name] = len(journal_names)
    return journal_names[name]

for ex in examples:
    ni = ex["metadata_journal_name_i"]
    nj = ex["metadata_journal_name_j"]
    ci = int(ex["metadata_citation_count_ij"])
    cj = int(ex["metadata_citation_count_ji"])
    if ci < MIN_FLOW:
        continue
    ii = get_idx(ni)
    ij = get_idx(nj)
    journal_labels[ni] = int(ex["metadata_label_i"])
    if nj not in journal_labels:
        journal_labels[nj] = int(ex["metadata_label_j"])
    edge_counts[(ii, ij)] = ci
    # Record reverse direction (may be 0)
    if (ij, ii) not in edge_counts:
        edge_counts[(ij, ii)] = cj

N = len(journal_names)                     # number of nodes
idx2name = {v: k for k, v in journal_names.items()}

# --- Build edge list with antisymmetric log-flow weights ---
# Only keep (i→j) where i < j OR where C[i,j] > C[j,i], to avoid double-counting
# Actually keep ALL directed edges for which we have data from the examples
directed_edges = []   # list of (i, j, f_ij)
seen_undirected = set()
for ex in examples:
    ni = ex["metadata_journal_name_i"]
    nj = ex["metadata_journal_name_j"]
    ci = int(ex["metadata_citation_count_ij"])
    cj = int(ex["metadata_citation_count_ji"])
    if ci < MIN_FLOW:
        continue
    ii = journal_names[ni]
    ij = journal_names[nj]
    key = (min(ii, ij), max(ii, ij))
    if key in seen_undirected:
        continue
    seen_undirected.add(key)
    # f(i→j) = log(C_ij + 1) - log(C_ji + 1): positive = net flow from i to j
    f_ij = np.log(ci + 1) - np.log(cj + 1)
    directed_edges.append((ii, ij, f_ij))

E = len(directed_edges)
print(f"Graph: {N} journals (nodes), {E} undirected edges")
print(f"Labels available: {len(journal_labels)} journals")
n_labeled_supp = sum(1 for v in journal_labels.values() if v == 1)
print(f"  Suppressed: {n_labeled_supp}, Clean: {len(journal_labels) - n_labeled_supp}")

## Step 2: Hodge Decomposition — Separating Gradient Flow from Curl

**HodgeRank** decomposes any antisymmetric edge flow $f \in \mathbb{R}^E$ into three orthogonal components:

$$f = f_{\text{grad}} + f_{\text{curl}} + f_{\text{harm}}$$

- **Gradient** $f_{\text{grad}}(i \to j) = s_j - s_i$: a consistent global ranking (node potentials $s$)
- **Curl** $f_{\text{curl}}$: cyclic inconsistencies — triangles where $A > B > C > A$
- **Harmonic** $f_{\text{harm}}$: topological loops not captured by triangles

**Key idea for manipulation detection:** A journal that artificially inflates its citation count by organizing citation rings will create cyclic inconsistencies in the citation flow — these show up as high **curl residuals**.

**Algorithm:**
1. Build signed incidence matrix $B \in \{-1, 0, +1\}^{E \times N}$ (head = +1, tail = −1)
2. Solve $L s = b$ where $L = B^T B$ (graph Laplacian) and $b = B^T f$ (flow divergence)
3. Gradient component: $f_{\text{grad}} = B s$
4. Curl residual per edge: $f_{\text{curl}}(i \to j) = f(i \to j) - (s_j - s_i)$
5. Node curl score: $\text{curl\_score}(v) = \frac{1}{\deg(v)} \sum_{e \ni v} |f_{\text{curl}}(e)|$

In [ ]:
# --- Build signed incidence matrix B (E × N) ---
# Convention: B[e, tail] = -1, B[e, head] = +1
rows_B, cols_B, vals_B = [], [], []
f_vec = np.zeros(E)
for e_idx, (ii, ij, f_ij) in enumerate(directed_edges):
    rows_B.extend([e_idx, e_idx])
    cols_B.extend([ii, ij])
    vals_B.extend([-1.0, 1.0])  # tail = -1, head = +1
    f_vec[e_idx] = f_ij

B = sp.csr_matrix((vals_B, (rows_B, cols_B)), shape=(E, N))

# --- Graph Laplacian L = B^T B and divergence b = B^T f ---
L = B.T @ B                      # (N × N) Laplacian
b = B.T @ f_vec                   # (N,) divergence vector

# --- Solve L * s = b (least-norm solution via pseudoinverse / regularization) ---
# L is singular (rank N-1), add small regularization to make it invertible
L_reg = L + HODGE_REG * sp.eye(N)
s, info = spla.cg(L_reg.tocsr(), b, maxiter=1000)
# Center potentials: set mean to zero
s = s - s.mean()

# --- Gradient and curl components ---
f_grad = B @ s                    # gradient part: f_grad[e] = s[head] - s[tail]
f_curl = f_vec - f_grad           # curl residual per edge

print(f"Hodge decomposition complete")
print(f"  ||f||^2        = {np.dot(f_vec, f_vec):.4f}")
print(f"  ||f_grad||^2   = {np.dot(f_grad, f_grad):.4f}  ({100*np.dot(f_grad,f_grad)/np.dot(f_vec,f_vec):.1f}%)")
print(f"  ||f_curl||^2   = {np.dot(f_curl, f_curl):.4f}  ({100*np.dot(f_curl,f_curl)/np.dot(f_vec,f_vec):.1f}%)")
print(f"  Node potential s: min={s.min():.3f}, max={s.max():.3f}")

## Step 3: Extract Node-Level Features

For each journal node $v$, we compute:

| Feature | Description |
|---------|-------------|
| `hodge_potential` | Node potential $s_v$ from Hodge decomposition (global ranking) |
| `curl_score` | Mean absolute curl residual over incident edges |
| `curl_max` | Maximum absolute curl residual over incident edges |
| `out_degree` | Number of outgoing citation edges |
| `in_degree` | Number of incoming citation edges |
| `net_flow` | Total outgoing − incoming citation count (raw asymmetry) |

The **curl score** is the key novel feature — it captures how much a journal's citation pattern deviates from a globally consistent ranking.

In [ ]:
# --- Aggregate edge-level curl residuals to node-level features ---
node_curl_abs = defaultdict(list)   # v -> [|f_curl(e)| for e incident to v]
node_out_count = defaultdict(int)   # v -> number of outgoing edges
node_in_count = defaultdict(int)    # v -> number of incoming edges
node_out_flow = defaultdict(float)  # v -> sum of f(v→j)
node_in_flow = defaultdict(float)   # v -> sum of f(j→v)

for e_idx, (ii, ij, f_ij) in enumerate(directed_edges):
    abs_curl = abs(f_curl[e_idx])
    # Both endpoints accumulate this edge's curl
    node_curl_abs[ii].append(abs_curl)
    node_curl_abs[ij].append(abs_curl)
    # Direction-specific degree and flow
    node_out_count[ii] += 1
    node_in_count[ij] += 1
    node_out_flow[ii] += f_ij
    node_in_flow[ij] += f_ij

# --- Build feature matrix for journals that have ground-truth labels ---
labeled_journals = [(name, label) for name, label in journal_labels.items()
                    if journal_names.get(name) is not None and
                       journal_names[name] in node_curl_abs]

X_rows, y_labels, names_labeled = [], [], []
for name, label in labeled_journals:
    v = journal_names[name]
    curls = node_curl_abs[v]
    hodge_s = float(s[v])
    curl_mean = np.mean(curls) if curls else 0.0
    curl_max  = np.max(curls)  if curls else 0.0
    out_deg   = node_out_count[v]
    in_deg    = node_in_count[v]
    net_f     = node_out_flow[v] - node_in_flow[v]
    X_rows.append([hodge_s, curl_mean, curl_max, out_deg, in_deg, net_f])
    y_labels.append(label)
    names_labeled.append(name)

X = np.array(X_rows, dtype=float)
y = np.array(y_labels, dtype=int)
feature_names = ["hodge_potential", "curl_score", "curl_max",
                 "out_degree", "in_degree", "net_flow"]

print(f"Feature matrix: {X.shape[0]} journals × {X.shape[1]} features")
print(f"Class balance: {y.sum()} suppressed, {(y==0).sum()} clean")
print(f"\nFeature means (suppressed vs clean):")
for i, fname in enumerate(feature_names):
    m1 = X[y==1, i].mean()
    m0 = X[y==0, i].mean()
    print(f"  {fname:20s}: suppressed={m1:+.3f}, clean={m0:+.3f}, diff={m1-m0:+.3f}")

## Step 4: Classification & Evaluation

We train a logistic regression classifier to predict journal suppression from the Hodge features, evaluated via **leave-one-out ROC-AUC** across stratified cross-validation folds.

We compare three feature sets:
1. **Hodge all**: All 6 features (novel method)
2. **Curl only**: Just `curl_score` + `curl_max` (the key Hodge features)
3. **Baseline** (degree only): `out_degree` + `in_degree` + `net_flow`

In [ ]:
from sklearn.dummy import DummyClassifier

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

cv = StratifiedKFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=SEED)
clf = LogisticRegression(max_iter=500, random_state=SEED, solver='lbfgs')

feature_sets = {
    "Hodge all (6 features)"    : list(range(6)),
    "Curl only (2 features)"    : [1, 2],          # curl_score, curl_max
    "Degree baseline (3 feat.)" : [3, 4, 5],        # out_deg, in_deg, net_flow
}

results = {}
for fname, fidx in feature_sets.items():
    Xs = X_scaled[:, fidx]
    scores = cross_val_score(clf, Xs, y, cv=cv, scoring='roc_auc')
    results[fname] = scores
    print(f"{fname:35s}: AUC = {scores.mean():.3f} ± {scores.std():.3f}")

# Full-data AUC (for visualization)
clf_full = LogisticRegression(max_iter=500, random_state=SEED, solver='lbfgs')
clf_full.fit(X_scaled, y)
y_score_full = clf_full.predict_proba(X_scaled)[:, 1]
auc_full = roc_auc_score(y, y_score_full)
print(f"\nFull-data AUC (train=test, optimistic): {auc_full:.3f}")

## Results & Visualization

The plots below show:
1. **Cross-validation AUC** comparison across feature sets
2. **Top journals by curl score** — do suppressed journals cluster at the top?
3. **Node potential vs curl score** scatter — suppressed journals highlighted

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# --- Plot 1: CV AUC bar chart ---
ax = axes[0]
labels_bar = ["Hodge\nall", "Curl\nonly", "Degree\nbaseline"]
means = [results[k].mean() for k in feature_sets]
stds  = [results[k].std()  for k in feature_sets]
colors = ['#2196F3', '#4CAF50', '#FF9800']
bars = ax.bar(labels_bar, means, yerr=stds, color=colors, alpha=0.8, capsize=5)
ax.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='Random')
ax.set_ylim(0, 1.05)
ax.set_ylabel("ROC-AUC")
ax.set_title(f"Cross-validation AUC ({N_CV_FOLDS}-fold)")
ax.legend(fontsize=9)
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, m + 0.02, f'{m:.3f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

# --- Plot 2: Top journals by curl score ---
ax = axes[1]
curl_scores = X[:, 1]  # curl_score feature
order = np.argsort(curl_scores)[::-1][:20]  # top 20 by curl
top_names   = [names_labeled[i][:20] for i in order]
top_curls   = [curl_scores[i] for i in order]
top_labels  = [y[i] for i in order]
bar_colors  = ['#F44336' if l == 1 else '#2196F3' for l in top_labels]
ax.barh(range(len(top_names)), top_curls[::-1], color=bar_colors[::-1], alpha=0.8)
ax.set_yticks(range(len(top_names)))
ax.set_yticklabels(top_names[::-1], fontsize=7)
ax.set_xlabel("Mean Curl Residual Score")
ax.set_title("Top 20 Journals by Curl Score\n(red = suppressed)")
# Add legend
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#F44336', label='Suppressed'),
                   Patch(color='#2196F3', label='Clean')], fontsize=8)

# --- Plot 3: Hodge potential vs curl score scatter ---
ax = axes[2]
mask_supp  = y == 1
mask_clean = y == 0
ax.scatter(X[mask_clean, 0], X[mask_clean, 1],
           c='#2196F3', alpha=0.6, s=40, label='Clean', zorder=2)
ax.scatter(X[mask_supp, 0], X[mask_supp, 1],
           c='#F44336', alpha=0.8, s=60, marker='*', label='Suppressed', zorder=3)
ax.set_xlabel("Hodge Potential (global ranking)")
ax.set_ylabel("Curl Score (anomaly signal)")
ax.set_title("Hodge Potential vs Curl Score\nper Journal")
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig("hodge_results.png", dpi=100, bbox_inches='tight')
plt.show()

# --- Summary table ---
print("\n" + "="*60)
print("SUMMARY: Hodge Decomposition for Citation Manipulation Detection")
print("="*60)
print(f"Network: {N} journals, {E} citation edges")
print(f"Labels: {y.sum()} suppressed, {(y==0).sum()} clean")
print()
print(f"{'Feature Set':<36} {'AUC mean':>10} {'AUC std':>10}")
print("-"*58)
for fname, scores in results.items():
    print(f"{fname:<36} {scores.mean():>10.3f} {scores.std():>10.3f}")
print("-"*58)
print(f"Fraction of flow explained by gradient: "
      f"{100*np.dot(f_grad,f_grad)/np.dot(f_vec,f_vec):.1f}%")
print(f"Fraction in curl component: "
      f"{100*np.dot(f_curl,f_curl)/np.dot(f_vec,f_vec):.1f}%")